## Identificacao

**Aluno:** *Lucas de Oliveira Santos*

---

**Dados:** Inside Airbnb — Rio de Janeiro, coleta 24/06/2026  
**Arquivo:** `listings.csv` (48.729 anúncios, 19 variáveis)

> ⚠️ Execute este notebook do início ao fim sem erros antes de submeter. Certifique-se de que `listings.csv` está na mesma pasta que este arquivo.

In [43]:
!pip install streamlit
!pip install pandas
!pip install numpy
!pip install altair

In [44]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


df = pd.read_csv("listings.csv.gz")

df["price_num"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

top_bairros = (
    df["neighbourhood_cleansed"]
    .value_counts()
    .head(10)
    .index
)

template = "plotly_white"

In [45]:
print("Dimensões:", df.shape)
display(df.head())
display(df[["price_num", "minimum_nights", "accommodates"]].describe())

Dimensões: (48713, 91)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,price_num
0,1025196153924029661,https://www.airbnb.com/rooms/1025196153924029661,20260624212719,2026-06-25,city scrape,Quartos disponíveis no Recreio,NaN,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,546603190,...,NaN,NaN,NaN,NaN,1,0,1,0,NaN,NaN
1,1697400829368754530,https://www.airbnb.com/rooms/1697400829368754530,20260624212719,2026-06-25,city scrape,Casa próxima a Floresta da Tijuca,Take a break to relax in this serene oasis in ...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,737983482,...,NaN,NaN,NaN,NaN,1,0,1,0,NaN,219.11
2,42874066,https://www.airbnb.com/rooms/42874066,20260624212719,2026-06-25,city scrape,Alugo uma vaga para sexo feminino Recreio,NaN,NaN,https://a0.muscache.com/pictures/43e779d6-58df...,341395950,...,NaN,NaN,NaN,NaN,1,0,1,0,NaN,1659.27
3,1543530193994981165,https://www.airbnb.com/rooms/1543530193994981165,20260624212719,2026-06-26,previous scrape,Barra Vista maravilhosa,"Kick back and relax in this calm, stylish space.",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,173836079,...,NaN,NaN,NaN,NaN,19,19,0,0,NaN,704.00
4,831750365913119274,https://www.airbnb.com/rooms/831750365913119274,20260624212719,2026-06-25,city scrape,Suíte Rio BR,Spacious and pleasant suite in a family home.,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,69238992,...,NaN,NaN,NaN,NaN,1,0,1,0,NaN,575.24


,price_num,minimum_nights,accommodates
count,44542.000000,48706.000000,48713.000000
mean,877.707421,3.139264,3.929177
std,4599.612177,13.706673,2.261917
min,5.540000,1.000000,1.000000
25%,296.710000,1.000000,2.000000
50%,452.670000,2.000000,4.000000
75%,761.537500,3.000000,5.000000
max,574013.000000,730.000000,16.000000


## Parte 1 - Perguntas de Investigacao

Formule duas perguntas respondíveis com os dados.  
Para cada uma: justifique a relevância para o observatório e indique as variáveis (1–2 frases).

### Pergunta 1

**Pergunta:** *Como os preços das hospedagens variam entre os principais bairros do Rio de Janeiro e essas diferenças permanecem quando comparamos acomodações com capacidade semelhante?*

**Relevância e variáveis:** *A análise permite identificar diferenças espaciais de preço e verificar se elas podem ser explicadas apenas pelo porte das acomodações. Serão utilizadas as variáveis neighbourhood_cleansed, price e accommodates.*

### Pergunta 2

**Pergunta:** *Os principais bairros do Rio de Janeiro apresentam perfis diferentes de exigência de estadia mínima?*

**Relevância e variáveis:** *A análise permite comparar como as regras de permanência mínima variam entre bairros e identificar áreas com maior presença relativa de ofertas voltadas a estadias curtas ou mais longas. Serão utilizadas as variáveis neighbourhood_cleansed e minimum_nights.*

## Parte 2 - Visualizacoes

Produza 3 a 4 gráficos que respondam às suas perguntas.  
Para **pelo menos dois**, justifique: (a) a escolha do tipo de gráfico e (b) as decisões de tratamento (filtragem, escala, exclusão de extremos).  
Todos os gráficos devem ter **título, eixos rotulados e legenda** quando houver mais de uma série.

In [46]:
# O preço no arquivo detalhado é texto: "$1,234.00"
df["price_num"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# Principais bairros definidos pelo número de anúncios
top_bairros = (
    df["neighbourhood_cleansed"]
    .value_counts()
    .head(10)
    .index
)

# Estilo comum
template = "plotly_white"

### Gráfico 1 — *Preço mediano das hospedagens nos 10 bairros com mais anúncios*

**Justificativa do tipo:** *O objetivo é comparar uma medida única entre categorias nominais. A orientação horizontal facilita a leitura dos nomes dos bairros e torna o ranking imediatamente perceptivel.*  
**Decisões de tratamento:** *Estou trazendo os 10 bairros com maior número de anúncios, evitando uma visualização com 154 categorias (bairros). Utilizei a mediana ao invés da média pois a distribuição de preços apresenta forte assimetria e valores extremos: A média geral é muito superior a mediana e existem anúncios com preços excepcionalmente altos. Anúncios sem preço informado foram excluídos do cálculo*

In [47]:
preco_bairro = (
    df[
        df["neighbourhood_cleansed"].isin(top_bairros)
        & df["price_num"].notna()
    ]
    .groupby("neighbourhood_cleansed", as_index=False)
    .agg(
        preco_mediano=("price_num", "median"),
        anuncios=("price_num", "size")
    )
    .sort_values("preco_mediano", ascending=True)
)

fig1 = px.bar(
    preco_bairro,
    x="preco_mediano",
    y="neighbourhood_cleansed",
    orientation="h",
    text="preco_mediano",
    hover_data={"anuncios": True, "preco_mediano": ":.2f"},
    labels={
        "preco_mediano": "Preço mediano por noite (R$)",
        "neighbourhood_cleansed": "Bairro",
        "anuncios": "Anúncios"
    },
    title="Preço mediano das hospedagens nos 10 bairros com mais anúncios"
)

fig1.update_traces(
    texttemplate="R$ %{text:,.0f}",
    textposition="outside"
)

fig1.update_layout(
    template=template,
    showlegend=False,
    xaxis_title="Preço mediano por noite (R$)",
    yaxis_title="",
    margin=dict(l=20, r=80, t=70, b=40)
)

fig1.show()

### Gráfico 2 — *Preço mediano por bairro para acomodações de capacidade semelhante*

**Justificativa do tipo:** *O gráfico de barras agrupadas permite comparar simultaneamente bairros e capacidades de hospedagem. A separação entre anúncios para 2, 4 e 6 hóspedes permite observar se as diferenças de preço entre bairros permanecem mesmo quando comparamos ofertas de porte semelhante.*  
**Decisões de tratamento:** *Foram selecionados os 10 bairros com maior oferta e as capacidades de 2, 4 e 6 hóspedes, que possuem grande número de observações e representam segmentos frequentes do conjunto de dados. Novamente foi utilizada a mediana para reduzir a influência dos preços extremos. Anúncios sem preço informado foram excluídos.*

In [48]:
# Gráfico 2
capacidades = [2, 4, 6]

preco_capacidade = (
    df[
        df["neighbourhood_cleansed"].isin(top_bairros)
        & df["accommodates"].isin(capacidades)
        & df["price_num"].notna()
    ]
    .groupby(
        ["neighbourhood_cleansed", "accommodates"],
        as_index=False
    )
    .agg(
        preco_mediano=("price_num", "median"),
        anuncios=("price_num", "size")
    )
)

# Ordem dos bairros pelo preço dos anúncios para 4 hóspedes
ordem_bairros = (
    preco_capacidade[
        preco_capacidade["accommodates"] == 4
    ]
    .sort_values("preco_mediano", ascending=False)
    ["neighbourhood_cleansed"]
    .tolist()
)

preco_capacidade["Capacidade"] = (
    preco_capacidade["accommodates"]
    .astype(str)
    + " hóspedes"
)

fig2 = px.bar(
    preco_capacidade,
    x="neighbourhood_cleansed",
    y="preco_mediano",
    color="Capacidade",
    barmode="group",
    category_orders={
        "neighbourhood_cleansed": ordem_bairros,
        "Capacidade": ["2 hóspedes", "4 hóspedes", "6 hóspedes"]
    },
    hover_data={
        "anuncios": True,
        "preco_mediano": ":.2f",
        "accommodates": False
    },
    labels={
        "neighbourhood_cleansed": "Bairro",
        "preco_mediano": "Preço mediano por noite (R$)"
    },
    title="Preço mediano por bairro para acomodações de capacidade semelhante"
)

fig2.update_layout(
    template=template,
    xaxis_title="Bairro",
    yaxis_title="Preço mediano por noite (R$)",
    legend_title="Capacidade",
    xaxis_tickangle=-35,
    margin=dict(t=70, b=100)
)

fig2.show()

### Gráfico 3 — *Perfil de estadia mínima nos 10 bairros com mais anúncios*

**Justificativa do tipo:** *Foi utilizado um gráfico de barras horizontais 100% empilhadas porque a pergunta envolve comparar a composição das políticas de estadia mínima entre bairros. Como cada barra representa 100% dos anúncios de um bairro, é possível comparar diretamente a participação relativa das diferentes faixas de permanência, independentemente da quantidade total de anúncios em cada região.*  
**Decisões de tratamento:** *A variável minimum_nights foi agrupada nas faixas 1 noite, 2–3 noites, 4–6 noites, 7–29 noites e 30 ou mais noites. O agrupamento foi adotado porque a variável apresenta forte assimetria e valores extremos de até 730 noites, tornando a comparação direta dos valores individuais pouco informativa. Foram novamente considerados os 10 bairros com maior oferta.*

In [49]:
# Preparação
df_min = df[
    df["neighbourhood_cleansed"].isin(top_bairros)
    & df["minimum_nights"].notna()
].copy()

df_min["perfil_estadia"] = pd.cut(
    df_min["minimum_nights"],
    bins=[0, 1, 3, 6, 29, float("inf")],
    labels=[
        "1 noite",
        "2–3 noites",
        "4–6 noites",
        "7–29 noites",
        "30+ noites"
    ]
)

perfil_bairro = (
    df_min
    .groupby(
        ["neighbourhood_cleansed", "perfil_estadia"],
        observed=True
    )
    .size()
    .reset_index(name="quantidade")
)

perfil_bairro["percentual"] = (
    perfil_bairro.groupby("neighbourhood_cleansed")["quantidade"]
    .transform(lambda x: x / x.sum() * 100)
)

fig3 = px.bar(
    perfil_bairro,
    y="neighbourhood_cleansed",
    x="percentual",
    color="perfil_estadia",
    orientation="h",
    barmode="stack",
    labels={
        "neighbourhood_cleansed": "Bairro",
        "percentual": "Participação dos anúncios (%)",
        "perfil_estadia": "Estadia mínima"
    },
    title="Perfil de estadia mínima nos 10 bairros com mais anúncios"
)

fig3.update_layout(
    template="plotly_white",
    xaxis_title="Participação dos anúncios (%)",
    yaxis_title="",
    legend_title="Estadia mínima",
    xaxis=dict(range=[0, 100]),
    margin=dict(l=20, r=20, t=70, b=40)
)

fig3.show()

### Gráfico 4 — *Participação de anúncios com estadia mínima de 7 noites ou mais*

**Justificativa do tipo:** *O gráfico de barras horizontais foi escolhido porque o objetivo é comparar uma proporção específica entre bairros. O ordenamento das barras facilita identificar quais áreas apresentam maior participação relativa de anúncios com exigência de permanência mais longa.*  
**Decisões de tratamento:** *Foi criada uma variável binária separando anúncios com estadia mínima inferior a 7 noites daqueles que exigem 7 noites ou mais. A análise é apresentada em percentual, e não em quantidade absoluta, para tornar comparáveis bairros com volumes diferentes de anúncios. O corte de 7 noites foi utilizado como uma forma simples e interpretável de distinguir anúncios que exigem pelo menos uma semana de permanência, sem assumir que isso represente necessariamente aluguel de longa duração.*

In [50]:
df_min["estadia_7_mais"] = df_min["minimum_nights"] >= 7

longa_bairro = (
    df_min
    .groupby("neighbourhood_cleansed", as_index=False)
    .agg(
        percentual_7_mais=("estadia_7_mais", "mean"),
        anuncios=("minimum_nights", "size")
    )
)

longa_bairro["percentual_7_mais"] *= 100

longa_bairro = longa_bairro.sort_values(
    "percentual_7_mais",
    ascending=True
)

fig4 = px.scatter(
    longa_bairro,
    x="percentual_7_mais",
    y="neighbourhood_cleansed",
    text="percentual_7_mais",
    labels={
        "percentual_7_mais": "Anúncios com estadia mínima de 7+ noites (%)",
        "neighbourhood_cleansed": "Bairro"
    },
    title="Participação de anúncios com estadia mínima de 7 noites ou mais"
)

fig4.update_traces(
    marker=dict(size=14),
    texttemplate="%{text:.1f}%",
    textposition="middle right"
)

fig4.update_layout(
    template="plotly_white",
    xaxis_title="Participação de anúncios (%)",
    yaxis_title="",
    showlegend=False,
    margin=dict(l=20, r=80, t=70, b=40)
)

fig4.update_yaxes(
    categoryorder="total ascending"
)

fig4.show()

### Dashboard — Capturas de Tela

Inclua aqui **pelo menos duas capturas de tela** do dashboard em execução.  
Para inserir uma imagem salva na mesma pasta: `![descrição](screenshot1.png)`

*(substitua screenshot1.png pelo nome real do arquivo)*

![descrição](streamlit_dash.png)

## Parte 3 - Conclusoes, Limitacoes e Reflexao Critica

Para cada pergunta: apresente a resposta suportada pelas evidências, pelo menos **duas limitações** dos dados que impedem generalizar, e uma investigação adicional que dados complementares permitiriam.  

Além disso, demonstre pensamento crítico sobre sua análise (ex.: alterar uma decisão de tratamento e verificar o efeito; apresentar explicação alternativa; verificar um número de forma independente).

## Parte 3 - Conclusões, Limitações e Reflexão Crítica

### Conclusões — Pergunta 1

**Pergunta:** Como os preços das hospedagens variam entre os principais bairros do Rio de Janeiro e essas diferenças permanecem quando comparamos acomodações com capacidade semelhante?

**Resposta:**  
Os resultados mostram diferenças relevantes de preço entre os principais bairros do Rio de Janeiro. Mesmo quando a comparação é feita entre acomodações com capacidade semelhante, o padrão permanece: bairros como Leblon, Ipanema e Barra da Tijuca apresentam preços medianos mais elevados, enquanto Centro, Santa Teresa e outros bairros apresentam valores menores.

**Limitações:**
1. A comparação por capacidade torna as acomodações mais semelhantes, mas não controla outras características importantes que podem influenciar o preço, como número de quartos, número de banheiros, comodidades, qualidade do imóvel, vista, proximidade de pontos turísticos e localização específica dentro de cada bairro.
2. Os dados representam anúncios ativos em uma única data de coleta. Os preços podem variar ao longo do ano em função de sazonalidade, feriados, grandes eventos e alterações realizadas pelos anfitriões, portanto os resultados não devem ser generalizados para todos os períodos.

**Investigação adicional:**  
Com dados do calendário do Airbnb, seria possível investigar a evolução dos preços ao longo do tempo e verificar se as diferenças entre bairros permanecem em diferentes períodos do ano. Também seria possível incorporar características como número de quartos, banheiros e comodidades para comparar imóveis ainda mais semelhantes.

### Conclusões — Pergunta 2

**Pergunta:** Os principais bairros do Rio de Janeiro apresentam perfis diferentes de exigência de estadia mínima?

**Resposta:**  
Os resultados mostram que estadias mínimas curtas predominam nos principais bairros analisados, especialmente anúncios que exigem uma noite ou entre duas e três noites. Entretanto, existem diferenças relevantes na participação de anúncios com exigências maiores de permanência

**Limitações:**
1. A variável `minimum_nights` informa apenas a quantidade mínima de noites exigida pelo anúncio e não permite identificar a motivação dessa regra. Portanto, não é possível concluir que anúncios com mínimos maiores estejam necessariamente associados a turismo de longa duração, aluguel residencial ou hospedagem corporativa.
2. A análise considera os 10 bairros com maior número de anúncios. Esse recorte melhora a legibilidade e a comparabilidade das visualizações, mas não representa todos os 154 bairros existentes na base e pode ocultar perfis diferentes em regiões com menor quantidade de anúncios.

**Investigação adicional:**  
O arquivo de calendário permitiria verificar se a exigência de estadia mínima varia ao longo do tempo e se anúncios com permanências mínimas maiores apresentam padrões diferentes de disponibilidade. Dados adicionais sobre reservas ou ocupação também permitiriam investigar se diferentes regras de permanência correspondem a diferentes perfis efetivos de uso da hospedagem.

### Reflexão Crítica

Durante a análise eu mudei algumas decisões conforme fui entendendo melhor os dados. No preço, por exemplo, percebi que a média ficava muito puxada por valores muito altos, então preferi usar a mediana para comparar os bairros.

Também achei que comparar só o preço entre bairros poderia ser meio injusto, porque bairros com imóveis maiores poderiam parecer mais caros só por isso. Por esse motivo, comparei também acomodações com capacidades parecidas, usando grupos de 2, 4 e 6 hóspedes. Mesmo assim, as diferenças entre alguns bairros continuaram aparecendo.

Cheguei a testar também a relação entre preço e avaliação dos hóspedes, mas as notas ficaram muito próximas entre si, quase sempre perto de 5. Como essa análise não acrescentou muita coisa, preferi descartar e focar em algo que mostrasse diferenças mais claras.

No caso de `minimum_nights`, também evitei usar só a média, porque existiam alguns valores muito altos. Achei mais fácil de interpretar separar em faixas de permanência e comparar as proporções entre os bairros.

In [51]:
# Código de suporte à reflexão crítica (análise de sensibilidade,
# verificação independente, comparação de cenários, etc.)
df = pd.read_csv('listings.csv')
print("\nPREÇO")
print("Mínimo:", df["price"].min())
print("Mediana:", df["price"].median())
print("Média:", df["price"].mean())
print("Máximo:", df["price"].max())

print("\nPERCENTIS DE PREÇO")
display(
    df["price"].quantile([0.50, 0.75, 0.90, 0.95, 0.99, 1.00])
)

print("\nPREÇO POR TIPO DE ACOMODAÇÃO")
display(
    df.groupby("room_type")["price"]
      .agg(["count", "mean", "median", "min", "max"])
      .sort_values("median", ascending=False)
)


PREÇO
Mínimo: 6.0
Mediana: 453.0
Média: 877.7035707071841
Máximo: 574013.0

PERCENTIS DE PREÇO


0.50       453.00
0.75       762.00
0.90      1400.00
0.95      2283.00
0.99      7578.88
1.00    574013.00
Name: price, dtype: float64


PREÇO POR TIPO DE ACOMODAÇÃO


,count,mean,median,min,max
room_type,,,,,
Hotel room,25,599.960000,582.0,150.0,1694.0
Entire home/apt,36153,964.611761,489.0,6.0,574013.0
Private room,7795,521.926748,300.0,6.0,183602.0
Shared room,584,258.238014,125.0,35.0,9275.0


## Parte 4 - IA e Reprodutibilidade

Liste as ferramentas de IA usadas. Para um resultado relevante, descreva em até 5 linhas: o que foi pedido, o que foi produzido e o que você verificou, corrigiu ou descartou.

**Ferramentas de IA usadas:**
- GPT-5.6 Sol (site) e Codex / GPT-6 Astra (software)

- **O que foi pedido:** Eu pedi apoio para que ele gerasse os códigos necessários para fazer a exploração inicial dos dados, assim eu não precisaria me preocupar com o código, e sim somente com o conteúdo da base de dados. Depois que eu entendi um pouco mais sobre o conteúdo dos dados, fui discutindo com ele até formular as perguntas que eu gostaria de responder. Após definir isso, comecei a montar os plots, e ele foi me ajudando a deixá-los mais legíveis. A parte de pegar toda a análise e gerar o Streamlit eu deleguei completamente para o GPT-6 Astra do Codex, pois era basicamente replicar o que já havia sido feito, mas em um dashboard.

- **O que foi produzido:** Código para exploração, sugestões de perguntas, transformações e visualizações em Plotly.

- **O que foi verificado / corrigido / descartado:** Eu conferi os resultados gerados, comparei os gráficos com os dados e descartei uma análise de preço x avaliações porque as diferenças eram muito pequenas. Também ajustei as perguntas finais e mantive apenas os gráficos que respondiam melhor ao problema.

**Minha percepção sobre o processo:** Eu particularmente gosto muito de escrever código, mas, em cenários como esse, em que o tempo era curto, é muito legal saber que temos ferramentas que conseguem abstrair toda a parte técnica e nos permitem focar apenas na análise e no que esses dados querem dizer para a gente.